In [ ]:
import yaml 

# Load the credentials YAML file
credential_path = '../credentials.yml' 
cfg = yaml.load(open(credential_path, "r"), Loader=yaml.Loader)
cfg

In [ ]:
# Create aws session & s3 client
import boto3
import awswrangler as wr

# Credentials
aws_access_key_id = cfg['aws']['aws_access_key_id']
aws_secret_access_key = cfg['aws']['aws_secret_access_key']

# create a authorized session using boto3 + credentials
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)
# create s3 client to interact with AWS S3
s3 = session.client(service_name='s3')

SA Metro GTFS
- Loading Mode: Initial Full Load + Incremental Load
    - Initial Full Load: Load the previous 10 versions of GTFS
    - Incremental Load:
        - Get the latest version in `landing zone` & latest version available from API.
        - Execute load if the API's latest version has not been ingested.  
- Partitioned Destination: `<landing_bucket>/gtfs/<FEED VERSION>`

In [ ]:
import requests
import os
from io import BytesIO
from zipfile import ZipFile

dest_path = 'destination'
os.makedirs(dest_path, exist_ok=True)

In [ ]:
# FULL LOAD
# 1. read data from SA Metro GTFS feed
# 2. upload feed version into landing bucket
base_url = "http://gtfs.adelaidemetro.com.au/v1"
latest_version_number = "static/latest/version.txt"
latest_version_feed = "static/latest/google_transit.zip"

In [ ]:
destination_bucket = 'cm-aws-s3-destination'

for version in range(1600, 1611):
    print("Version: ", version)
    version_feed_url = f"{base_url}/static/{version}/google_transit.zip"
    print(version_feed_url)
    response = requests.get(version_feed_url, stream=True)

    dest_path = f'k1/lybui/gtfs/{version}/google_transit_{version}.zip'

    s3.put_object(
        Bucket=destination_bucket,
        Key=dest_path,
        Body=response.content
    )

In [ ]:
# INCREMENTAL LOAD
# 1. Get: API's latest version & AWS S3's versions
# 2. Compare & logging whether the latest version is already ingested
# 3. Execute the ingestion if not already ingested

# Practice: define script into tasks / functions
# e.g., Func1: Get API Latest version
# Func2: Get AWS S3 versions and compare if API latest version is already ingested -> True/False
# Func3: Execute the API latest version ingestion if Func2 return False

In [ ]:
def get_api_latest_version(base_url, latest_version_number):
    version_feed_url = f"{base_url}/{latest_version_number}"
    response = requests.get(version_feed_url)
    return int(response.content)

latest_api_version = get_api_latest_version(base_url=base_url, latest_version_number=latest_version_number)
latest_api_version

In [ ]:
def get_aws_s3_latest_version(s3_bucket, s3_folder_path):
    response = s3.list_objects_v2(
        Bucket=s3_bucket,
        Prefix=s3_folder_path   
    )
    versions = []
    contents = response.get('Contents')

    for result in contents:
        key = result.get("Key")
        version_number = os.path.basename(key).split("_")[-1].split(".")[0]
        versions.append(version_number)

    latest_version = max(versions)
    return int(latest_version)

s3_bucket = 'cm-aws-s3-destination'
s3_folder_path = f'k1/lybui/'

response = get_aws_s3_latest_version(s3_bucket, s3_folder_path)
response

In [22]:
def ingest_latest_api_to_aws_s3(s3_bucket, s3_folder_path, base_url, latest_version_number):
    # Get latest API version on AWS S3 
    latest_s3_version = get_aws_s3_latest_version(s3_bucket=s3_bucket, s3_folder_path=s3_folder_path)

    # Get latest API version
    latest_api_version = get_api_latest_version(base_url=base_url, latest_version_number=latest_version_number)

    if latest_s3_version < latest_api_version:
        for version in range(latest_s3_version, latest_api_version + 1):
            print("Version: ", version)
            version_feed_url = f"{base_url}/static/{version}/google_transit.zip"
            print(version_feed_url)
            response = requests.get(version_feed_url, stream=True)

            dest_path = f'k1/lybui/gtfs/{version}/google_transit_{version}.zip'

            s3.put_object(
                Bucket=destination_bucket,
                Key=dest_path,
                Body=response.content
            )
            print(f"Uploaded version {version} to S3 bucket!")
    else: 
        print("The current API version is the latest!")

In [23]:
s3_bucket = 'cm-aws-s3-destination'
s3_folder_path = f'k1/lybui/'
base_url = "http://gtfs.adelaidemetro.com.au/v1"
latest_version_number = "static/latest/version.txt"

ingest_latest_api_to_aws_s3(s3_bucket=s3_bucket, 
                            s3_folder_path=s3_folder_path, 
                            base_url=base_url, 
                            latest_version_number=latest_version_number)

k1/lybui/gtfs/1600/google_transit_1600.zip
k1/lybui/gtfs/1601/google_transit_1601.zip
k1/lybui/gtfs/1602/google_transit_1602.zip
k1/lybui/gtfs/1603/google_transit_1603.zip
k1/lybui/gtfs/1604/google_transit_1604.zip
k1/lybui/gtfs/1605/google_transit_1605.zip
k1/lybui/gtfs/1606/google_transit_1606.zip
k1/lybui/gtfs/1607/google_transit_1607.zip
k1/lybui/gtfs/1608/google_transit_1608.zip
k1/lybui/gtfs/1609/google_transit_1609.zip
k1/lybui/gtfs/1610/google_transit_1610.zip
k1/lybui/gtfs/1611/google_transit_1611.zip
The current API version is the latest!
